# Phase 1: Design & Prompts Demo

This notebook demonstrates the Phase 1 design components for the Newsletter Generation System.

## Overview

Phase 1 establishes:
- **Shared Agent State**: Typed container for all workflow data
- **Tool I/O Contracts**: Input/output schemas for all 8 tools
- **Production Prompts**: Style-aware prompts for 4 agent departments
- **Middleware Hooks**: Chain of responsibility for production concerns
- **Style Profiles**: Customizable writing styles
- **Evaluation Framework**: Rubrics, evaluators, and metrics

## 1. Agent State Schema

In [1]:
# Import the state schema
import sys
sys.path.insert(0, '..')

from config.state import (
    AgentState,
    RunRequest,
    Task,
    Fact,
    Citation,
    ConfidenceLevel,
    check_stop_conditions,
    slice_for_researcher,
    slice_for_writer,
    slice_for_editor,
    slice_for_diagrammer,
)

print("Agent State Schema loaded successfully!")

Agent State Schema loaded successfully!


In [2]:
# Create a sample run request
run_request = RunRequest(
    topic="Apache Kafka Fundamentals",
    brief="Introduction to Kafka architecture for Vietnamese engineers",
    requirements=[
        "Explain producer-consumer model",
        "Cover partition and replication",
        "Include Vietnamese company use cases",
    ],
    style_profile_id="neutral-concise",
)

print("Run Request:")
print(run_request.model_dump_json(indent=2))

Run Request:
{
  "topic": "Apache Kafka Fundamentals",
  "brief": "Introduction to Kafka architecture for Vietnamese engineers",
  "requirements": [
    "Explain producer-consumer model",
    "Cover partition and replication",
    "Include Vietnamese company use cases"
  ],
  "style_profile_id": "neutral-concise",
  "sources": [],
  "publish": false,
  "max_iterations": 5,
  "thread_id": null
}


In [3]:
# Initialize an agent state
state = AgentState(
    run_id="demo-run-001",
    workflow_status="running",
    current_agent="researcher",
)

# Add a task
task = Task(
    task_id="TASK_001",
    topic=run_request.topic,
    brief=run_request.brief,
    requirements=run_request.requirements,
    style_profile_id=run_request.style_profile_id,
)
state.tasks.append(task)
state.current_task_id = task.task_id

print(f"State initialized with task: {task.topic}")
print(f"Current agent: {state.current_agent}")

State initialized with task: Apache Kafka Fundamentals
Current agent: researcher


In [4]:
# Add sample facts and citations
state.facts.append(Fact(
    claim_id="CLAIM_001",
    text="Apache Kafka can process millions of events per second",
    source_ids=["SRC_001", "SRC_002"],
    confidence=ConfidenceLevel.HIGH,
    category="technical",
))

state.citations.append(Citation(
    source_id="SRC_001",
    url="https://kafka.apache.org/documentation/",
    title="Apache Kafka Documentation",
    domain="kafka.apache.org",
    domain_trust="high",
    is_primary=True,
))

state.citations.append(Citation(
    source_id="SRC_002",
    url="https://confluent.io/blog/kafka-performance/",
    title="Kafka Performance at Scale",
    domain="confluent.io",
    domain_trust="high",
    is_primary=False,
))

print(f"Facts: {len(state.facts)}")
print(f"Citations: {len(state.citations)}")

Facts: 1
Citations: 2


In [5]:
# Demonstrate context slicing
researcher_context = slice_for_researcher(state)
print("Researcher's Context Slice:")
for key, value in researcher_context.items():
    print(f"  {key}: {type(value).__name__}")

Researcher's Context Slice:
  topic: str
  brief: str
  requirements: list
  style_constraints: list
  existing_citations: list
  existing_facts: list


In [6]:
# Check stop conditions
should_stop, reason = check_stop_conditions(state)
print(f"Should stop: {should_stop}")
print(f"Reason: {reason}")

Should stop: False
Reason: continue


## 2. Tool I/O Contracts

In [7]:
from tools.contracts import (
    TOOL_CONTRACTS,
    get_tool_contract,
    validate_tool_input,
    WebSearchInput,
    WebSearchOutput,
    CitationBuilderInput,
    DrawioGenerateInput,
    DiagramType,
)

print("Available Tools:")
for name, contract in TOOL_CONTRACTS.items():
    hitl = "(HITL)" if contract["hitl_gated"] else ""
    print(f"  - {name} {hitl}")

Available Tools:
  - web.search 
  - web.fetch (HITL)
  - local.search 
  - citation.builder 
  - drawio.generate 
  - drawio.export 
  - seo.readability 
  - plagiarism.scan (HITL)


In [8]:
# Validate a web search input
search_input = WebSearchInput(
    query="Apache Kafka 2024 performance benchmarks",
    k=5,
    include_domains=["kafka.apache.org", "confluent.io"],
)

print("Web Search Input:")
print(search_input.model_dump_json(indent=2))

Web Search Input:
{
  "query": "Apache Kafka 2024 performance benchmarks",
  "k": 5,
  "include_domains": [
    "kafka.apache.org",
    "confluent.io"
  ],
  "exclude_domains": []
}


In [9]:
# Create a diagram generation input
diagram_input = DrawioGenerateInput(
    intent="Show Kafka producer-consumer message flow",
    diagram_type=DiagramType.DATA_FLOW,
    title="Kafka Message Flow",
    nodes=[
        {"id": "producer", "label": "Producer App", "type": "rectangle"},
        {"id": "broker", "label": "Kafka Broker", "type": "cylinder"},
        {"id": "consumer", "label": "Consumer App", "type": "rectangle"},
    ],
    edges=[
        {"source": "producer", "target": "broker", "label": "publish"},
        {"source": "broker", "target": "consumer", "label": "consume"},
    ],
    width=650,
)

print("Diagram Generation Input:")
print(diagram_input.model_dump_json(indent=2))

Diagram Generation Input:
{
  "intent": "Show Kafka producer-consumer message flow",
  "diagram_type": "data_flow",
  "title": "Kafka Message Flow",
  "nodes": [
    {
      "id": "producer",
      "label": "Producer App",
      "type": "rectangle",
      "style": null
    },
    {
      "id": "broker",
      "label": "Kafka Broker",
      "type": "cylinder",
      "style": null
    },
    {
      "id": "consumer",
      "label": "Consumer App",
      "type": "rectangle",
      "style": null
    }
  ],
  "edges": [
    {
      "source": "producer",
      "target": "broker",
      "label": "publish",
      "style": "solid"
    },
    {
      "source": "broker",
      "target": "consumer",
      "label": "consume",
      "style": "solid"
    }
  ],
  "width": 650
}


In [10]:
# Get contract details for web.fetch (HITL gated)
fetch_contract = get_tool_contract("web.fetch")
print("web.fetch contract:")
print(f"  HITL Gated: {fetch_contract['hitl_gated']}")
print(f"  Condition: {fetch_contract.get('hitl_condition', 'N/A')}")
print(f"  Description: {fetch_contract['description']}")

web.fetch contract:
  HITL Gated: True
  Condition: domain_trust == 'unknown' or domain_trust == 'low'
  Description: Fetch and extract content from a URL. HITL required for unknown domains.


## 3. Style Profiles

In [11]:
from config.style_profiles import (
    StyleProfile,
    get_profile,
    list_profiles,
    inject_profile_into_prompt,
    DEFAULT_PROFILES,
)

print("Available Style Profiles:")
for profile_id in list_profiles():
    print(f"  - {profile_id}")

Available Style Profiles:
  - neutral_concise
  - technical-deep
  - neutral-concise
  - beginner-friendly


In [12]:
# Load the default profile
profile = get_profile("neutral-concise")
print(f"Profile: {profile.name}")
print(f"Voice: {profile.voice}")
print(f"Target Audience: {profile.target_audience}")
print(f"\nCitation Policy:")
print(f"  Min sources per claim: {profile.citation_policy.min_sources_per_claim}")
print(f"  Require primary source: {profile.citation_policy.require_primary_source}")

Profile: Neutral Concise
Voice: neutral, confident, helpful
Target Audience: Vietnamese tech professionals (data engineers, architects)

Citation Policy:
  Min sources per claim: 2
  Require primary source: True


In [13]:
# Generate prompt string from profile
prompt_string = profile.to_prompt_string()
print("Style Profile for Prompt Injection:")
print(prompt_string)

Style Profile for Prompt Injection:

## Style Profile: Neutral Concise

### Voice & Tone
- Voice: neutral, confident, helpful
- Tone: professional, clear, concise

### Structure Requirements
- Sections: hook → summary → sections → key-takeaways → CTA
- Word count: 800-1200 words
- Key takeaways: 3-5 items

### Readability Targets
- Grade level: 12 (±1)
- Sentence length: 12-20 words
- Max passive voice: 20%

### Citation Policy
- Minimum sources per claim: 2
- Require primary source: True
- Max single domain: 60%

### DO NOT USE
- sensational claims
- vague sources
- clickbait headlines
- excessive jargon without explanation

### Target Audience
Vietnamese tech professionals (data engineers, architects) (intermediate level)



In [14]:
# Demonstrate profile injection
template = """You are a writer.

{STYLE_PROFILE}

Write content based on the above style."""

injected = inject_profile_into_prompt(template, profile)
print("Injected Prompt (first 500 chars):")
print(injected[:500] + "...")

Injected Prompt (first 500 chars):
You are a writer.


## Style Profile: Neutral Concise

### Voice & Tone
- Voice: neutral, confident, helpful
- Tone: professional, clear, concise

### Structure Requirements
- Sections: hook → summary → sections → key-takeaways → CTA
- Word count: 800-1200 words
- Key takeaways: 3-5 items

### Readability Targets
- Grade level: 12 (±1)
- Sentence length: 12-20 words
- Max passive voice: 20%

### Citation Policy
- Minimum sources per claim: 2
- Require primary source: True
- Max single domain: 60...


## 4. Middleware Chain

In [15]:
from config.middleware import (
    create_default_middleware_chain,
    MiddlewareContext,
    LimitsConfig,
    HITLConfig,
    MiddlewarePriority,
)

# Create default middleware chain
chain = create_default_middleware_chain(
    limits_config=LimitsConfig(
        max_model_calls_per_run=30,
        max_tool_calls_per_run=50,
    ),
    hitl_config=HITLConfig(
        gate_unknown_domains=True,
        auto_approve_domains=["kafka.apache.org", "confluent.io"],
    ),
)

print("Middleware Chain (execution order):")
for i, mw in enumerate(chain.middleware, 1):
    print(f"  {i}. {mw.name} (priority: {mw.priority})")

Middleware Chain (execution order):
  1. CallLimitsMiddleware (priority: MiddlewarePriority.LIMITS)
  2. PIIRedactionMiddleware (priority: MiddlewarePriority.PII_REDACTION)
  3. HITLGateMiddleware (priority: MiddlewarePriority.HITL_GATE)
  4. ToolRetryMiddleware (priority: MiddlewarePriority.RETRY)
  5. ModelFallbackMiddleware (priority: MiddlewarePriority.FALLBACK)
  6. SummarizationMiddleware (priority: MiddlewarePriority.SUMMARIZATION)
  7. StyleProfileMiddleware (priority: MiddlewarePriority.STYLE_PROFILE)
  8. CanonicalCitationsMiddleware (priority: MiddlewarePriority.CITATIONS)


In [16]:
# Create a middleware context
ctx = MiddlewareContext(
    run_id="demo-run-001",
    agent_name="researcher",
    tool_name="web.search",
    input_data={"query": "Kafka performance"},
)

print("Middleware Context:")
print(f"  Run ID: {ctx.run_id}")
print(f"  Agent: {ctx.agent_name}")
print(f"  Tool: {ctx.tool_name}")
print(f"  Model calls: {ctx.model_calls}")
print(f"  Tool calls: {ctx.tool_calls}")

Middleware Context:
  Run ID: demo-run-001
  Agent: researcher
  Tool: web.search
  Model calls: 0
  Tool calls: 0


## 5. Evaluation Framework

In [17]:
from evals import (
    get_seed_dataset,
    NEWSLETTER_TOPICS,
    CitationRubric,
    ReadabilityRubric,
    StyleRubric,
    evaluate_with_rubrics,
    create_evaluator_suite,
)

# Get seed dataset
dataset = get_seed_dataset()
print(f"Seed Dataset: {dataset.name}")
print(f"Total entries: {len(dataset)}")

# Show category distribution
categories = {}
for entry in dataset:
    categories[entry.category] = categories.get(entry.category, 0) + 1

print("\nBy Category:")
for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count}")

Seed Dataset: newsletter_topics_v1
Total entries: 20

By Category:
  analytics: 1
  architecture: 2
  batch: 2
  cloud: 1
  mlops: 1
  quality: 1
  storage: 2
  streaming: 9
  transformation: 1


In [18]:
# Show sample dataset entries
print("Sample Dataset Entries:")
for entry in list(dataset)[:3]:
    print(f"\n  Topic: {entry.topic}")
    print(f"  Category: {entry.category}")
    print(f"  Difficulty: {entry.difficulty}")
    print(f"  Expected diagrams: {entry.expected_diagram_types}")

Sample Dataset Entries:

  Topic: Apache Kafka Fundamentals
  Category: streaming
  Difficulty: easy
  Expected diagrams: ['architecture', 'data_flow']

  Topic: Kafka Connect Deep Dive
  Category: streaming
  Difficulty: medium
  Expected diagrams: ['data_flow', 'architecture']

  Topic: Kafka Streams vs Flink
  Category: streaming
  Difficulty: hard
  Expected diagrams: ['architecture']


In [19]:
# Demonstrate rubric evaluation with sample data
sample_draft = """
# Apache Kafka Fundamentals

Apache Kafka is a distributed streaming platform.
<!-- claim_id: CLAIM_001 -->

## Summary

Learn about Kafka's architecture and core concepts.

## Architecture

Kafka uses a distributed commit log architecture.
<!-- claim_id: CLAIM_002 -->

## Key Takeaways

- Kafka is highly scalable
- Kafka provides durability through replication
- Kafka supports real-time streaming

## What's Next?

Try setting up your first Kafka cluster!
"""

sample_facts = [
    {"claim_id": "CLAIM_001", "source_ids": ["SRC_001", "SRC_002"]},
    {"claim_id": "CLAIM_002", "source_ids": ["SRC_001"]},  # Only 1 source
]

sample_citations = [
    {"source_id": "SRC_001", "domain": "kafka.apache.org", "is_primary": True},
    {"source_id": "SRC_002", "domain": "confluent.io", "is_primary": False},
]

sample_profile = profile.model_dump()

In [20]:
# Run rubric evaluation
results = evaluate_with_rubrics(
    draft=sample_draft,
    citations=sample_citations,
    facts=sample_facts,
    style_profile=sample_profile,
)

print("Rubric Evaluation Results:")
for name, result in results.items():
    print(f"\n{name}:")
    print(f"  Status: {result.status.value}")
    print(f"  Score: {result.score:.2f}")
    print(f"  Message: {result.message}")
    if result.issues:
        print(f"  Issues: {len(result.issues)}")
        for issue in result.issues[:2]:
            print(f"    - [{issue.severity.value}] {issue.description}")

Rubric Evaluation Results:

citation_rubric:
  Status: warning
  Score: 0.90
  Message: Citation density: 100%, 2 unique domains
  Issues: 1
    - [major] Found 1 claims with <2 sources

readability_rubric:
  Status: warning
  Score: 0.70
  Message: Grade level: 8.3, Words: 66
  Issues: 2
    - [major] Word count 66 below minimum 800
    - [major] Reading level 8.3 outside target 12±1

style_rubric:
  Status: warning
  Score: 0.85
  Message: Found 5 sections, 4 missing
  Issues: 1
    - [major] Missing required sections: hook, sections, key-takeaways, CTA

diagram_rubric:
  Status: pass
  Score: 1.00
  Message: Generated 0/0 diagrams


In [21]:
# Create evaluator suite
evaluators = create_evaluator_suite()
print("Evaluator Suite:")
for evaluator in evaluators:
    print(f"  - {evaluator.name}: {evaluator.description}")

Evaluator Suite:
  - citation_density: Measures percentage of claims with proper citations
  - source_independence: Measures domain diversity of citations
  - reading_level: Evaluates Flesch-Kincaid grade level
  - fact_count: Evaluates minimum number of extracted facts
  - source_count: Evaluates minimum number of unique sources
  - diagram_inclusion: Evaluates presence of expected diagrams
  - structure_compliance: Evaluates presence of required sections


## 6. Production Prompts Preview

In [23]:
import yaml
from pathlib import Path

prompts_dir = Path("../config/prompts")

print("Production Prompt Files:")
for prompt_file in sorted(prompts_dir.glob("*.yaml")):
    print(f"  - {prompt_file.name}")

Production Prompt Files:
  - _global_system.yaml
  - diagrammer.yaml
  - editor.yaml
  - eic.yaml
  - research_strategist.yaml
  - research_technical.yaml
  - researcher.yaml
  - writer.yaml


In [24]:
# Load and preview researcher prompt
researcher_prompt_path = prompts_dir / "researcher.yaml"
if researcher_prompt_path.exists():
    with open(researcher_prompt_path) as f:
        researcher_prompt = yaml.safe_load(f)
    
    print("Researcher Prompt (first 1000 chars):")
    print(researcher_prompt.get("prompt", "")[:1000] + "...")
else:
    print("Researcher prompt file not found")

Researcher Prompt (first 1000 chars):
# RESEARCHER - Evidence Gathering Specialist

You are the **Researcher** for a Vietnamese technical newsletter system. Your mission is to
gather deep, accurate technical information with proper citations that enable high-quality
content creation.

## YOUR MISSION

Transform a topic and brief into a comprehensive research package:
- Verified facts with confidence levels
- Properly cited sources (>=2 independent per claim)
- Coverage of all brief requirements
- Gaps clearly marked as TODO

## SEARCH STRATEGY

1. **Initial Broad Search**: Start with the main topic to understand the landscape
2. **Targeted Deep Dives**: Search specific subtopics from the brief requirements
3. **Verification Searches**: Find independent sources to cross-verify key claims
4. **Gap Filling**: Search for any uncovered requirements

### Query Formulation Tips
- Use specific technical terms, not vague descriptions
- Include year qualifiers for recent information (e.g., "Kafk

## 7. Definition of Done Checklist

### Phase 1 Deliverables:

- [x] **Typed Agent State** (`config/state.py`)
  - AgentState with all required fields
  - RunRequest input schema
  - Context slicing functions
  - Stop condition checks

- [x] **Tool I/O Contracts** (`tools/contracts.py`)
  - 8 tools with Pydantic schemas
  - HITL gate conditions
  - Validation helpers

- [x] **Agent Configuration** (`config/agents.yaml`)
  - 4 departments defined
  - Context slices documented
  - Handoff connections
  - Global settings

- [x] **Production Prompts** (`config/prompts/`)
  - Global system prompt
  - Researcher, Writer, Editor, Diagrammer prompts
  - Style profile injection

- [x] **Middleware Plan** (`config/middleware.py`)
  - 8 middleware classes
  - Execution order defined
  - Chain manager

- [x] **Style Profiles** (`config/style_profiles/`)
  - StyleProfile schema
  - 3 default profiles
  - Injection utilities

- [x] **Evaluation Framework** (`evals/`)
  - Seed dataset (20 topics)
  - Quality rubrics
  - Evaluators for LangSmith
  - Metrics aggregation

In [25]:
print("Phase 1 Design & Prompts - COMPLETE")
print("="*50)
print("\nNext Steps (Phase 2):")
print("  1. Implement middleware stack with real handlers")
print("  2. Wire tools to agents")
print("  3. Create HITL checkpoint simulation")
print("  4. Run integration tests")
print("  5. Set up LangSmith evaluation runs")

Phase 1 Design & Prompts - COMPLETE

Next Steps (Phase 2):
  1. Implement middleware stack with real handlers
  2. Wire tools to agents
  3. Create HITL checkpoint simulation
  4. Run integration tests
  5. Set up LangSmith evaluation runs
